# Семинар 5. REINFORCE руками: стратегия для CartPole

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/05-policy-gradient/seminar/seminar.ipynb)

На лекции REINFORCE учил одно число $p$ — вероятность → в коридоре, где агент не различает клеток. Сегодня вы пишете REINFORCE сами и учите им стратегию для CartPole: здесь агент видит четыре числа, и вероятность → зависит от них. Формулы — с лекции; новое одно: предпочтение действия считается по состоянию (раздел 2).

| | Что делаем | Раздел лекции | Минут |
|---|---|---|---|
| 1 | Softmax и шаг по $\nabla \log \pi$ | 3. Стратегия с параметрами: softmax | 12 |
| 2 | Новая среда: CartPole и стратегия из четырёх весов | 2–3 | 20 |
| 3 | REINFORCE на CartPole | 4. REINFORCE | 28 |
| 4 | Вычитаем среднее | 5. Вычитаем среднее | 25 |
| 5 | Что дальше | — | 5 |

Ячейки с `# TODO` — каждая на одну-шесть строк. Под каждой сразу идёт проверка: она должна пройти без ошибок — тогда двигайтесь дальше. Задача со звёздочкой ★ — по желанию.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"], check=True)

In [ ]:
# Служебный модуль: рисунки тележки, кривые обучения, карта стратегии. В Colab скачивается.
import pathlib, urllib.request
REPO = "https://raw.githubusercontent.com/IlyaChichkanov/Reinforcement-learning/main/"
path = "05-policy-gradient/seminar/cartpole_plots.py"
for d in ["", "../../05-policy-gradient/seminar"]:
    if pathlib.Path(d, "cartpole_plots.py").exists():
        sys.path.append(d)
        break
else:
    urllib.request.urlretrieve(REPO + path, "cartpole_plots.py")

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from IPython.display import display
from cartpole_plots import draw_cartpole, plot_lengths, plot_policy_map, plot_vs, animate_cartpole

## 1. Softmax и шаг по $\nabla \log \pi$

Повторим две формулы раздела 3 лекции — на них держится весь семинар. Стратегия задаётся предпочтениями $\theta_a$, по числу на каждое действие, а вероятности получаются формулой softmax:

$$
\pi(a) \;=\; \frac{e^{\theta_a}}{\sum_b e^{\theta_b}} ,
$$

где $a$ — действие, чью вероятность считаем, а сумма в знаменателе идёт по всем действиям $b$.

**TODO 1.1. Softmax.** Напишите `softmax(theta)` — две строки. Сначала вычтите из всех предпочтений наибольшее: вероятности от этого не изменятся (на лекции: прибавка одного числа ко всем предпочтениям умножает числитель и знаменатель на одно и то же), а `np.exp` не переполнится на больших числах вроде 1000.

In [ ]:
def softmax(theta):
    """Предпочтения θ (массив) -> вероятности действий π."""
    theta = np.asarray(theta, dtype=float)
    # TODO 1.1: две строки — z = θ − max θ; затем e^z, делённое на сумму e^z
    raise NotImplementedError


# Проверка 1.1 — пример из лекции и два свойства
p = softmax([1.0, 0.0, 0.0])
assert np.allclose(p, [0.576, 0.212, 0.212], atol=1e-3), \
    f"θ = (1, 0, 0) даёт (e, 1, 1) / (e + 2) ≈ (0.58, 0.21, 0.21), у вас {p.round(3)}"
assert np.allclose(softmax([5.0, 6.0]), softmax([0.0, 1.0])), \
    "прибавка одного числа ко всем предпочтениям не меняет вероятностей: (5, 6) и (0, 1) должны дать одно и то же"
big = softmax([1000.0, 0.0])
assert np.all(np.isfinite(big)) and np.allclose(big, [1.0, 0.0]), \
    f"при θ = (1000, 0) получилось {big}: вычтите наибольшее предпочтение перед np.exp"
print("OK: softmax(1, 0, 0) =", p.round(2))

Чтобы сделать сыгранное действие $a$ вероятнее, предпочтения сдвигают по градиенту $\log \pi(a)$. На лекции его вывели для softmax:

$$
\nabla_\theta \log \pi(a) \;=\; \vec e_a - \vec\pi ,
$$

где $\vec e_a$ — вектор с единицей на месте $a$ и нулями на остальных местах, а $\vec\pi$ — вектор вероятностей всех действий.

**TODO 1.2. Градиент логарифма.** `grad_log_pi(theta, a)` — одна строка. Вектор $\vec e_a$ — это `np.eye(len(theta))[a]`.

In [ ]:
def grad_log_pi(theta, a):
    """∇_θ log π(a) для softmax: вектор e_a − π той же длины, что θ."""
    # TODO 1.2: одна строка
    raise NotImplementedError


# Проверка 1.2 — вопрос 3 лекции: вероятности (0.5, 0.3, 0.2), сыграли B (номер 1), шаг 0.5
theta = np.log([0.5, 0.3, 0.2])
g = grad_log_pi(theta, 1)
assert np.isclose(g.sum(), 0.0), f"компоненты e_a − π в сумме дают 1 − 1 = 0, у вас {g.sum():.3f}"
after = softmax(theta + 0.5 * g)
assert np.allclose(after, [0.39, 0.43, 0.18], atol=0.01), \
    f"на лекции после шага вышло (0.39, 0.43, 0.18), у вас {after.round(2)}"
print("OK: после шага по e_B − π вероятности", after.round(2))

**Одной строкой:** softmax превращает предпочтения в вероятности, а шаг по $\vec e_a - \vec\pi$ делает сыгранное действие вероятнее.

## 2. Новая среда: CartPole и стратегия из четырёх весов

**Задача.** На тележке на шарнире стоит шест. Агент толкает тележку влево или вправо и должен как можно дольше удерживать шест от падения. Это среда `CartPole-v1`, которую мы видели в лекции 1. Опишем её по чек-листу:

* **Состояние** — четыре числа $\vec s = (x, \dot x, \varphi, \dot\varphi)$: положение тележки $x$ (центр рельса — 0), её скорость $\dot x$, угол шеста от вертикали $\varphi$ в радианах (плюс — шест наклонён вправо) и угловая скорость шеста $\dot\varphi$.
* **Действия** — два: 0 — толкнуть тележку ←, 1 — толкнуть →.
* **Награда** — +1 за каждый шаг, пока шест стоит.
* **Конец эпизода** — шест отклонился больше чем на 12° ($|\varphi| > 0.21$), тележка уехала за край рельса ($|x| > 2.4$) или прошло 500 шагов. Поэтому return при $\gamma = 1$ — число шагов, которое простоял шест, не больше 500.

Мерка всего семинара — **сколько шагов простоял шест**.

In [ ]:
env = gym.make("CartPole-v1")
s0, _ = env.reset(seed=0)
print("наблюдение на старте (x, ẋ, φ, φ̇):", s0.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
draw_cartpole(s0, ax=axes[0], title="старт: шест почти вертикален")
draw_cartpole([0.5, 0.0, 0.15, 0.8], ax=axes[1], action=1, title="шест падает вправо, толчок →")
plt.show()

**Стратегия.** В коридоре датчик не различал клеток, поэтому предпочтение → было одним числом $\theta$, а предпочтение ← закрепили нулём. Здесь агент видит состояние, и предпочтение → будем считать по нему — взвешенной суммой четырёх чисел:

$$
h(\vec s) \;=\; \vec w \cdot \vec s \;=\; w_1 x + w_2 \dot x + w_3 \varphi + w_4 \dot\varphi ,
$$

где $\vec w = (w_1, w_2, w_3, w_4)$ — четыре веса, их и будем учить. Предпочтение ← по-прежнему 0, поэтому вероятность → считается той же формулой, что в коридоре:

$$
p(\vec s) \;=\; \pi(\rightarrow \mid \vec s) \;=\; \frac{e^{h(\vec s)}}{1 + e^{h(\vec s)}} , \qquad \pi(\leftarrow \mid \vec s) \;=\; 1 - p(\vec s) .
$$

Пример: $\vec w = (0, 0, 1, 1)$, шест наклонён и падает вправо, $\vec s = (0, 0, 0.1, 0.5)$. Тогда $h = 0.1 + 0.5 = 0.6$ и $p = e^{0.6} / (1 + e^{0.6}) \approx 0.65$: толчок → вероятнее.

**Градиент.** На лекции для коридора: производная $\log \pi(a)$ по предпочтению → равна $1 - p$, если сыгран →, и $-p$, если ←. Здесь предпочтение → — это $h(\vec s)$, а $h$ зависит от весов линейно: $\partial h / \partial w_i = s_i$, то есть $\nabla_{\vec w}\, h = \vec s$. По правилу производной сложной функции

$$
\nabla_{\vec w} \log \pi(a \mid \vec s) \;=\; \frac{\partial \log \pi(a \mid \vec s)}{\partial h}\; \nabla_{\vec w}\, h \;=\; \big([a = \rightarrow] - p(\vec s)\big)\, \vec s ,
$$

где $[a = \rightarrow]$ равно 1, если сыгран →, и 0, если ←. Действие → в CartPole имеет номер 1, поэтому в коде это `(a - p) * s`.

**TODO 2.1.** `prob_right(w, s)` — одна строка через `softmax` из раздела 1: предпочтения двух действий — `[0.0, w @ s]` (`w @ s` — скалярное произведение), нужна вероятность действия 1.

**TODO 2.2.** `grad_log_pi_w(w, s, a)` — одна строка по формуле выше.

In [ ]:
def prob_right(w, s):
    """Вероятность толчка → в состоянии s: p = e^h / (1 + e^h), где h = w · s."""
    # TODO 2.1: одна строка
    raise NotImplementedError


def grad_log_pi_w(w, s, a):
    """∇_w log π(a | s) = ([a = →] − p(s)) · s; a — 0 (←) или 1 (→)."""
    # TODO 2.2: одна строка
    raise NotImplementedError


# Проверка 2.1–2.2 — пример из текста
w_demo, s_demo = np.array([0.0, 0.0, 1.0, 1.0]), np.array([0.0, 0.0, 0.1, 0.5])
p_demo = prob_right(w_demo, s_demo)
assert np.isclose(prob_right(np.zeros(4), s_demo), 0.5), "при w = 0 предпочтения обоих действий равны 0, значит p = 0.5"
assert np.isclose(p_demo, 0.646, atol=1e-3), f"h = 0.6, p = e^0.6 / (1 + e^0.6) ≈ 0.646, у вас {p_demo:.3f}"
assert np.allclose(grad_log_pi_w(w_demo, s_demo, 1), (1 - p_demo) * s_demo), "после → градиент равен (1 − p) · s"
assert np.allclose(grad_log_pi_w(w_demo, s_demo, 0), -p_demo * s_demo), "после ← градиент равен −p · s"
assert prob_right(w_demo + 0.1 * grad_log_pi_w(w_demo, s_demo, 1), s_demo) > p_demo, "шаг по градиенту после → должен поднять p"
assert prob_right(w_demo + 0.1 * grad_log_pi_w(w_demo, s_demo, 0), s_demo) < p_demo, "шаг по градиенту после ← должен опустить p"
print(f"OK: p = {p_demo:.2f}; градиент после → {grad_log_pi_w(w_demo, s_demo, 1).round(3)}")

**TODO 2.3. Эпизод.** Функция `play` играет один эпизод стратегией с весами $\vec w$ и возвращает списки состояний, действий и наград. Каркас готов, допишите тело цикла: вероятность → в текущем состоянии, действие 1 с этой вероятностью (`int(rng.random() < p)`), шаг среды и запись $s$, $a$, $r$ в списки.

Под функцией — две проверки. При $\vec w = 0$ стратегия случайная: $p = 0.5$ в любом состоянии. Веса $(0, 0, 10, 10)$ толкают тележку туда, куда наклонён и падает шест, и держат его заметно дольше.

In [ ]:
def play(env, w, rng, seed=None):
    """Один эпизод стратегией с весами w. Возвращает списки состояний, действий и наград."""
    s, _ = env.reset(seed=seed)
    states, actions, rewards = [], [], []
    while True:
        # TODO 2.3: p = prob_right(w, s); a = int(rng.random() < p);
        #           s2, r, terminated, truncated, _ = env.step(a); записать s, a, r в три списка
        raise NotImplementedError
        s = s2
        if terminated or truncated:
            return states, actions, rewards


# Проверка 2.3
rng = np.random.default_rng(0)
states, actions, rewards = play(env, np.zeros(4), rng, seed=0)
assert len(states) == len(actions) == len(rewards), "на каждом шаге записывается ровно одно состояние, одно действие и одна награда"
assert set(actions) <= {0, 1} and all(r == 1.0 for r in rewards), "действия — 0 или 1, награда за каждый шаг +1"
lengths_random = [len(play(env, np.zeros(4), rng, seed=i)[2]) for i in range(100)]
lengths_push = [len(play(env, np.array([0.0, 0.0, 10.0, 10.0]), rng, seed=i)[2]) for i in range(20)]
assert 15 < np.mean(lengths_random) < 30, \
    f"случайная стратегия держит шест в среднем около 20 шагов, у вас {np.mean(lengths_random):.1f}"
assert np.mean(lengths_push) > 100, \
    (f"веса (0, 0, 10, 10) держат шест в среднем {np.mean(lengths_push):.0f} шагов, а должны больше 100: "
     "проверьте, что действие 1 (→) выпадает с вероятностью p, а не 1 − p")
print(f"OK: случайная стратегия (w = 0) — в среднем {np.mean(lengths_random):.1f} шага, "
      f"веса (0, 0, 10, 10) — {np.mean(lengths_push):.0f}")

**Одной строкой:** стратегия для CartPole — четыре веса $\vec w$; вероятность → считается как в коридоре, только предпочтение → теперь равно $\vec w \cdot \vec s$.

## 3. REINFORCE на CartPole

**Задача.** Подобрать веса $\vec w$ по сыгранным эпизодам так, чтобы шест стоял дольше. Формулы — из раздела 4 лекции: после каждого эпизода

$$
\vec w \;\leftarrow\; \vec w + \alpha \sum_{t=0}^{T-1} \gamma^t\, G_t\, \nabla_{\vec w} \log \pi(a_t \mid \vec s_t) , \qquad G_t \;=\; \sum_{k=t}^{T-1} \gamma^{k-t}\, r_k ,
$$

где $T$ — длина эпизода, $\alpha$ — шаг, $\gamma$ — дисконт, $G_t$ — дисконтированная сумма наград с шага $t$ до конца эпизода. Берём $\gamma = 0.99$. В CartPole каждая награда +1, поэтому $G_t = 1 + 0.99 + 0.99^2 + \dots$: если после шага $t$ шест простоит ещё долго, $G_t$ близко к $1 / (1 - 0.99) = 100$.

**TODO 3.1. Награды до конца эпизода.** `returns_to_go(rewards, gamma)` — массив всех $G_t$. Считайте с конца: $G_{T-1} = r_{T-1}$, затем $G_t = r_t + \gamma\, G_{t+1}$ — так вся сумма считается за один проход.

In [ ]:
def returns_to_go(rewards, gamma):
    """G_t = r_t + γ r_{t+1} + γ² r_{t+2} + … для каждого шага t эпизода."""
    G = np.zeros(len(rewards))
    acc = 0.0
    # TODO 3.1: цикл по t с конца — for t in reversed(range(len(rewards))): acc = r_t + γ · acc; G[t] = acc
    raise NotImplementedError
    return G


# Проверка 3.1 — считается руками
G3 = returns_to_go([1.0, 1.0, 1.0], 0.5)
assert np.allclose(G3, [1.75, 1.5, 1.0]), f"G_2 = 1, G_1 = 1 + 0.5 · 1 = 1.5, G_0 = 1 + 0.5 · 1.5 = 1.75; у вас {G3}"
G500 = returns_to_go([1.0] * 500, 0.99)
assert np.isclose(G500[-1], 1.0) and np.isclose(G500[0], (1 - 0.99**500) / 0.01), \
    f"у эпизода из 500 шагов G_0 = (1 − 0.99^500) / 0.01 ≈ 99.3, у вас {G500[0]:.1f}"
print(f"OK: G для наград (1, 1, 1) при γ = 0.5 — {G3}; для 500 шагов CartPole G_0 = {G500[0]:.1f}")

**TODO 3.2. Шаг REINFORCE.** `reinforce_step(w, states, actions, rewards, alpha, gamma, b=0.0)` — сумма по шагам эпизода и сдвиг весов. Параметр $b$ пригодится в разделе 4, там вес шага станет $G_t - b$; пока $b = 0$, и вес — просто $G_t$.

Проверка — эпизод из двух шагов, который считается руками: $\vec w = 0$ (значит, $p = 0.5$), состояния $\vec s_0 = (0, 0, 0.1, 0)$ и $\vec s_1 = (0, 0, 0, 0.2)$, действия → и ←, награды 1 и 1, $\gamma = 0.5$, $\alpha = 1$. Тогда $G_0 = 1.5$, $G_1 = 1$ и

$$
\vec w_{\text{new}} \;=\; 1 \cdot 1.5 \cdot (1 - 0.5)\, \vec s_0 \;+\; 0.5 \cdot 1 \cdot (0 - 0.5)\, \vec s_1 \;=\; 0.75\, \vec s_0 - 0.25\, \vec s_1 \;=\; (0,\ 0,\ 0.075,\ -0.05) .

In [ ]:
def reinforce_step(w, states, actions, rewards, alpha, gamma, b=0.0):
    """Новые веса после одного эпизода: w + α Σ_t γ^t (G_t − b) ∇ log π(a_t | s_t)."""
    G = returns_to_go(rewards, gamma)
    total = np.zeros_like(w)
    for t in range(len(rewards)):
        # TODO 3.2: одна строка — прибавить к total слагаемое γ^t · (G[t] − b) · grad_log_pi_w(w, states[t], actions[t])
        raise NotImplementedError
    return w + alpha * total


# Проверка 3.2 — эпизод из двух шагов из текста
toy = dict(states=[np.array([0.0, 0.0, 0.1, 0.0]), np.array([0.0, 0.0, 0.0, 0.2])],
           actions=[1, 0], rewards=[1.0, 1.0], alpha=1.0, gamma=0.5)
w_toy = reinforce_step(np.zeros(4), **toy)
assert np.allclose(w_toy, [0, 0, 0.075, -0.05]), \
    f"0.75 · s_0 − 0.25 · s_1 = (0, 0, 0.075, −0.05), у вас {w_toy}; не забыли множитель γ^t?"
w_toy_b = reinforce_step(np.zeros(4), **toy, b=1.0)
assert np.allclose(w_toy_b, [0, 0, 0.025, 0.0]), \
    f"с b = 1 веса шагов 1.5 − 1 = 0.5 и 1 − 1 = 0, ответ (0, 0, 0.025, 0); у вас {w_toy_b}"
print("OK: шаг на игрушечном эпизоде", w_toy)

**TODO 3.3. Обучение.** Каркас `train` готов: он играет эпизоды и записывает, сколько шагов простоял шест. Допишите одну строку — шаг REINFORCE после эпизода. Когда `baseline=True`, вычитается среднее $b$ (раздел 4), иначе $b = 0$.

Шаг $\alpha = 0.01$, 400 эпизодов. Запустим обучение 10 раз с разными `seed`, чтобы видеть разброс между запусками. Мерка запуска — средняя длина его последних 50 эпизодов. 10 запусков считаются около 30 секунд.

In [ ]:
def train(n_episodes=400, alpha=0.01, gamma=0.99, baseline=False, seed=0):
    """REINFORCE на CartPole. Возвращает длины эпизодов, выученные веса и итоговое b."""
    env = gym.make("CartPole-v1")
    rng = np.random.default_rng(seed)
    w = np.zeros(4)
    b, n_steps = 0.0, 0                  # среднее G_t по всем шагам прошлых эпизодов — понадобится в разделе 4
    lengths = []
    for episode in range(n_episodes):
        states, actions, rewards = play(env, w, rng, seed=seed * 10_000 + episode)
        # TODO 3.3: одна строка — w = reinforce_step(...) с b, если baseline=True, и с 0.0 иначе
        raise NotImplementedError
        if baseline:
            b, n_steps = update_baseline(b, n_steps, returns_to_go(rewards, gamma))   # раздел 4
        lengths.append(len(rewards))
    return np.array(lengths), w, b


runs_plain = [train(seed=s) for s in range(10)]
lengths_plain = np.array([lengths for lengths, _, _ in runs_plain])
last_plain = lengths_plain[:, -50:].mean(axis=1)
print("средняя длина последних 50 эпизодов по запускам:", last_plain.round(0))
print(f"медиана {np.median(last_plain):.0f}; держат шест в среднем 400 шагов и больше: {(last_plain >= 400).sum()} из 10")

# Проверка 3.3
assert np.median(last_plain) > 100, \
    f"после обучения шест должен стоять заметно дольше, чем у случайной стратегии (около 20 шагов); медиана {np.median(last_plain):.0f}"

plot_lengths(lengths_plain, color="#DD8452", title="REINFORCE, вес шага $G_t$")
plt.show()

Все запуски научились держать шест дольше случайной стратегии (около 20 шагов): медиана средней длины последних 50 эпизодов — 252. Но запуски сильно разошлись — от 109 до 489 шагов, и 400 шагов и больше держат только 2 из 10.

**Почему так выходит.** В CartPole каждая награда +1, поэтому каждое $G_t$ положительно: от 1 у последнего шага до почти 100 у шагов, после которых шест простоял ещё долго. Шаг с положительным весом поднимает вероятность сыгранного действия, каким бы оно ни было, — ровно как с последним толчком в вопросе 5 лекции. Удачный толчок от неудачного отличает только разница в $G_t$: последние толчки перед падением получают $G_t$ около 1–10, но и их вероятность шаг поднимает. Эта разница теряется в общем сдвиге «вверх для всех действий эпизода», и каждый запуск учится медленно и по-своему.

**Одной строкой:** REINFORCE учит стратегию для CartPole по одним сыгранным эпизодам, но с весом $G_t > 0$ запуски сильно расходятся.

## 4. Вычитаем среднее

**Задача.** Сделать так, чтобы шаг поднимал вероятность только тех действий, после которых шест простоял дольше обычного, и опускал остальные. Способ — из раздела 5 лекции: вес шага $G_t - b$, где $b$ — среднее $G_t$ по всем шагам всех прошлых эпизодов. Число $b$ не зависит от выбора действия, поэтому средний шаг не меняется ($\mathbb{E}[b\, \nabla \log \pi] = 0$), а общий сдвиг «вверх для всех» исчезает.

**TODO 4.1. Среднее на ходу.** `update_baseline(b, n, G)` добавляет к среднему $b$ (посчитанному по $n$ прошлым шагам) все $G_t$ только что сыгранного эпизода, по одному, как в лекции 3: $n \leftarrow n + 1$, $b \leftarrow b + (G_t - b) / n$. Функция `train` вызывает её уже после шага REINFORCE — так $b$ в шаге не зависит от действий текущего эпизода.

In [ ]:
def update_baseline(b, n, G):
    """Среднее на ходу: добавить к среднему b по n прошлым шагам все G_t нового эпизода. Возвращает (b, n)."""
    for g in G:
        # TODO 4.1: две строки — n += 1; b += (g − b) / n
        raise NotImplementedError
    return b, n


# Проверка 4.1 — считается руками
assert np.allclose(update_baseline(0.0, 0, [1.0, 2.0, 3.0]), (2.0, 3)), "среднее чисел 1, 2, 3 — это 2, шагов 3"
assert np.allclose(update_baseline(2.0, 3, [6.0]), (3.0, 4)), "к трём шагам со средним 2 добавили 6: (1 + 2 + 3 + 6) / 4 = 3"
print("OK: среднее на ходу")

Те же 10 запусков, тот же шаг $\alpha = 0.01$; различие — только `baseline=True`. Считается около 50 секунд.

In [ ]:
runs_base = [train(baseline=True, seed=s) for s in range(10)]
lengths_base = np.array([lengths for lengths, _, _ in runs_base])
last_base = lengths_base[:, -50:].mean(axis=1)
print("средняя длина последних 50 эпизодов по запускам:", last_base.round(0))
print(f"медиана {np.median(last_base):.0f}; держат шест в среднем 400 шагов и больше: {(last_base >= 400).sum()} из 10")
print("b в конце обучения по запускам:", np.round([b for _, _, b in runs_base], 0))

# Проверка 4.2
assert (last_base >= 400).sum() >= 7, \
    "с вычитанием среднего большинство запусков должно держать шест 400 шагов и больше: проверьте update_baseline и вес G_t − b в train"

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
plot_lengths(lengths_plain, ax=axes[0], color="#DD8452", title="вес шага $G_t$")
plot_lengths(lengths_base, ax=axes[1], color="#55A868", title="вес шага $G_t - b$")
plt.tight_layout(); plt.show()

С вычитанием среднего все 10 запусков держат шест в среднем 400 шагов и больше (от 439 до 489), медиана — 475. Без вычитания таких было 2 из 10.

**Почему так выходит.** К концу обучения $b \approx 75$. Толчки за несколько шагов до падения получают $G_t$ около 1–10, и их вес $G_t - b$ сильно отрицательный, порядка −70: вероятность таких толчков в этих состояниях падает. Толчки, после которых шест простоял ещё долго, получают $G_t$ около 100 и вес порядка +25. Без вычитания у тех же последних толчков вес был +1…+10, и их вероятность росла.

**TODO 4.2. Как держит шест выученная стратегия.** Возьмите веса лучшего запуска и сыграйте эпизод без случайности: в каждом состоянии — действие с большей вероятностью (→, если $p \ge 0.5$, иначе ←). Соберите все состояния эпизода в список `states_best`, функция `animate_cartpole` сделает из них GIF.

In [ ]:
w_best = runs_base[int(np.argmax(last_base))][1]
s, _ = env.reset(seed=123)
states_best = [s]
while True:
    # TODO 4.2: три строки — a = 1, если prob_right(w_best, s) >= 0.5, иначе 0;
    #           s, r, terminated, truncated, _ = env.step(a); добавить s в states_best
    raise NotImplementedError
    if terminated or truncated:
        break

print(f"шест простоял {len(states_best) - 1} шагов")
assert len(states_best) - 1 >= 400, "без случайности лучший запуск должен держать шест 400 шагов и больше"
display(animate_cartpole(states_best))

Что выучила стратегия, видно по весам и по карте: при тележке в центре ($x = \dot x = 0$) цвет показывает вероятность → для каждого угла $\varphi$ и угловой скорости $\dot\varphi$ шеста. Красное — толчок →, синее — толчок ←, чёрная линия — $p = 0.5$.

In [ ]:
print("веса (x, ẋ, φ, φ̇):", w_best.round(1))
plot_policy_map(w_best, title="выученная стратегия: p(→) при x = ẋ = 0")
plt.show()

Веса при $\varphi$ и $\dot\varphi$ положительны: если шест наклонён или падает вправо, вероятность → растёт. Граница $p = 0.5$ почти совпадает с пунктиром — правилом, которое в лекции 1 придумал человек: толкать тележку туда, куда падает шест ($\dot\varphi > 0$ — толчок →). Небольшой наклон границы означает, что учитывается и угол: при шесте, наклонённом вправо, толчок → выбирается чуть раньше. Обучение по одним наградам привело к тому же правилу, что придумал человек.

**★ TODO 4.3. Подбор шага α.** При каких $\alpha$ работает каждый вариант? Для $\alpha \in \{0.003,\ 0.01,\ 0.03\}$ обучите оба варианта по 5 запусков и запишите в `results` медиану средней длины последних 50 эпизодов. Считается около двух минут.

In [ ]:
alphas = [0.003, 0.01, 0.03]
results = {"вес G_t": [], "вес G_t − b": []}
# TODO 4.3★: для каждого α из alphas и каждого варианта (baseline=False — "вес G_t", baseline=True — "вес G_t − b")
#            обучите 5 запусков train(alpha=α, baseline=..., seed=s) и добавьте в results[...] медиану
#            средних длин их последних 50 эпизодов
raise NotImplementedError

for name, values in results.items():
    print(name, np.round(values, 0))
plot_vs(alphas, results, xlabel="шаг α", ylabel="длина последних 50 эпизодов", title="медиана по 5 запускам")
plt.show()

**Вопрос.** При каких $\alpha$ каждый вариант держит шест дольше? Что происходит с отдельными запусками при самом большом шаге?

**Одной строкой:** вычитание среднего превращает вес шага в «дольше или короче обычного простоял шест после этого действия» — и REINFORCE на CartPole учится устойчиво.

## 5. Что дальше

* Сегодня вы написали REINFORCE целиком — softmax, градиент логарифма, награды до конца эпизода, шаг после эпизода и вычитание среднего — и выучили им стратегию для CartPole из четырёх весов.
* Дальше в курсе (Deep Policy Gradient) вместо четырёх весов будет нейросеть: предпочтения действий выдаёт сеть, а $\nabla \log \pi$ считает PyTorch. Формула шага останется той же.
* **Квиз недели 5** — на [странице квизов](https://ilyachichkanov.github.io/Reinforcement-learning/); [песочница лекции 5](https://ilyachichkanov.github.io/Reinforcement-learning/lecture05.html) — коридор.
* **Домашка 2** — дедлайн 3 ноября, 23:59 МСК.
* Решение этого семинара — в [`solutions/week05`](https://github.com/IlyaChichkanov/Reinforcement-learning/blob/main/solutions/week05/seminar.ipynb). Сначала попробуйте сами.